# Поправка долей по маршрутам

CPU. Проверяем один заранее заданный рецепт. Для каждого маршрута оцениваем ошибку доли Session6 на предыдущих исторических прогнозах, объединяя все восемь горизонтов. Переносим четверть относительной ошибки с ограничением ±5%. После поправки приводим каждый горизонт к исходному объёму Session6.

На двух предтестовых неделях сравниваем CatBoost depth 4 до и после поправки. На каждой неделе и метамодель, и коэффициенты используют только более ранние OOF. Локальный test оцениваем после фиксации рецепта, до переобучения поправки. Октябрьский test уже изучался.

Конкурсный CSV формируется из готового родительского сабмита и коэффициентов на всей доступной истории честных прогнозов Session6. Базовые модели и метамодель для конкурсных прогнозов уже рассчитаны в родительском сабмите. Используйте только готовый full-history CSV с объёмом Session6. Ввод и вывод можно задать переменными BLEND_PACKAGE и ROUTE_BIAS_PARENT. По умолчанию родительский CSV лежит рядом с ноутбуком и называется route_bias_parent.csv.


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path(os.environ.get('BLEND_PACKAGE', '/root/experiments/session6_blend_server'))
OOF_PATH = None
TEST_PATH = None
OUT = ROOT / 'output_route_bias_cpu'
OUT.mkdir(parents=True, exist_ok=True)

BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
GROUP_KEYS = ['cutoff', 'horizon']
KEYS = GROUP_KEYS + ['route_id']
COLUMNS = KEYS + BASES + ['y', 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
THREADS = 2
SEED = 42

def read_panel(path, name, start, end):
    archived = ROOT / 'blend_inputs' / 'blend_best_output' / name
    if path is None and archived.is_file():
        path = archived
    if path is None:
        candidates = [p for p in ROOT.rglob(name) if OUT not in p.parents
                      and set(COLUMNS).issubset(pq.ParquetFile(p).schema.names)]
        if len(candidates) != 1:
            raise ValueError(f'Укажите путь явно для {name}: {candidates}')
        path = candidates[0]
    frame = pd.read_parquet(path, columns=COLUMNS)
    frame['cutoff'] = pd.to_datetime(frame.cutoff)
    expected = pd.date_range(start, end, inclusive='left') + pd.Timedelta(hours=10, minutes=30)
    actual = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    counts = frame.groupby(GROUP_KEYS).size()
    if (not actual.equals(expected) or frame.duplicated(KEYS).any()
            or set(frame.horizon.unique()) != set(range(1, 9))
            or len(counts) != len(expected)*8 or not counts.eq(1000).all()
            or not np.isfinite(frame[COLUMNS[3:]].to_numpy()).all()
            or frame.y.lt(0).any() or frame.session6.lt(0).any()):
        raise ValueError(f'Некорректная панель: {path}')
    print(path, len(frame))
    return frame.reset_index(drop=True)

PARENT = Path(os.environ.get('ROUTE_BIAS_PARENT', str(ROOT / 'route_bias_parent.csv')))
REQUESTS_PATH = ROOT / 'blend_inputs/session6/test_solo_track.parquet'
STRENGTH = .25
MAX_CHANGE = .05


In [ ]:
def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level

def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def fit_meta(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix, frame.y)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    model = CatBoostRegressor(depth=4, iterations=220, loss_function='MAE', learning_rate=.035,
                              l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                              task_type='CPU', thread_count=THREADS)
    model.fit(features, (frame.y.to_numpy(dtype=float)-baseline)/level, sample_weight=level/level.mean())
    return model, weights

def predict_meta(frame, fitted):
    model, weights = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    return with_volume(frame, baseline + np.clip(model.predict(features), -.5, .5)*level)

def score_mix(frame, shape, weight):
    prediction = with_volume(frame, (1-weight)*frame.session6.to_numpy() + weight*shape)
    daily = pd.DataFrame([dict(day=str(day.date()), **metric(group.y, prediction[group.index]))
                          for day, group in frame.groupby('cutoff')])
    summary = dict(**metric(frame.y, prediction), daily_mean=daily.score.mean(),
                   daily_p90=daily.score.quantile(.9), daily_worst=daily.score.max())
    return summary, daily, prediction


In [ ]:
def fit_route_adjustment(history):
    volume = history.groupby(GROUP_KEYS)[['session6','y']].transform('sum')
    if volume.le(0).any().any():
        raise ValueError('Нулевой исторический объём')
    matched = history.y.to_numpy(dtype=float) * volume.session6.to_numpy() / volume.y.to_numpy()
    route = history.assign(matched=matched).groupby('route_id')[['matched','session6']].sum()
    factor = 1 + (STRENGTH * (route.matched.div(route.session6.where(route.session6>0)) - 1)).clip(-MAX_CHANGE,MAX_CHANGE).fillna(0.)
    return factor.rename('factor')

def adjust(frame, baseline, factors):
    scale = frame.route_id.map(factors).fillna(1.).to_numpy()
    anchor = np.asarray(baseline,dtype=np.float64)
    return with_volume(frame, anchor*scale, anchor=anchor)

def score_pair(frame, baseline, factors, period, fold):
    rows = []
    daily_rows = []
    adjusted = adjust(frame,baseline,factors)
    for variant, forecast in [('control',baseline), ('route_adjusted',adjusted)]:
        summary = metric(frame.y, forecast)
        daily = pd.DataFrame([dict(day=str(day.date()), **metric(group.y,forecast[group.index]))
                              for day,group in frame.groupby('cutoff')])
        rows.append(dict(period=period, fold=fold, variant=variant, **summary,
                         daily_mean=daily.score.mean(), daily_p90=daily.score.quantile(.9), daily_worst=daily.score.max()))
        daily_rows.append(daily.assign(period=period,fold=fold,variant=variant))
    return rows,daily_rows,adjusted


In [ ]:
oof = read_panel(OOF_PATH, 'base_oof.parquet', '2025-09-20', '2025-10-18').astype({'y':'float64','session6':'float64'})
rows = []
daily_rows = []
for boundary, end in [('2025-10-04','2025-10-11'), ('2025-10-11','2025-10-18')]:
    train = oof.loc[oof.cutoff+pd.to_timedelta(oof.horizon*30,unit='m') < pd.Timestamp(boundary)].reset_index(drop=True)
    val = oof.loc[(oof.cutoff >= boundary) & (oof.cutoff < end)].reset_index(drop=True)
    fitted = fit_meta(train)
    factors = fit_route_adjustment(train)
    for period, frame in [('train_oof',train),('val',val)]:
        result,daily,_ = score_pair(frame,predict_meta(frame,fitted),factors,period,boundary)
        rows.extend(result)
        daily_rows.extend(daily)

validation = pd.DataFrame(rows)
display(validation)
validation.to_csv(OUT/'validation_metrics.csv',index=False)
(OUT/'frozen_recipe.json').write_text(json.dumps(dict(strength=STRENGTH,max_change=MAX_CHANGE,group='route_id',test_used_for_selection=False),indent=2))


In [ ]:
test = read_panel(TEST_PATH, 'test_predictions.parquet', '2025-10-18', '2025-11-01').astype({'y':'float64','session6':'float64'})
fitted = fit_meta(oof)
factors = fit_route_adjustment(oof)
for period, frame in [('train_oof',oof),('test',test)]:
    result,daily,_ = score_pair(frame,predict_meta(frame,fitted),factors,period,'2025-10-18')
    rows.extend(result)
    daily_rows.extend(daily)

metrics = pd.DataFrame(rows)
metrics.to_csv(OUT/'metrics.csv',index=False)
pd.concat(daily_rows,ignore_index=True).to_csv(OUT/'daily_metrics.csv',index=False)
display(metrics[metrics.fold.eq('2025-10-18')])
print('Train OOF — оценка на обучающих данных метамодели.')
print('Test оценён до переобучения поправки на всей истории.')


In [ ]:
if PARENT.is_file():
    requests = pd.read_parquet(REQUESTS_PATH, columns=['id','route_id','timestamp'])
    parent = pd.read_csv(PARENT)
    if (len(requests)!=8000 or len(parent)!=8000 or list(parent.columns)!=['id','y_pred']
            or not requests.id.is_unique or not parent.id.is_unique
            or set(requests.id)!=set(parent.id) or not np.isfinite(parent.y_pred).all() or parent.y_pred.lt(0).any()):
        raise ValueError('Неверные ключи или прогнозы родительского сабмита')
    query = requests.merge(parent,on='id',validate='one_to_one',sort=False)
    final_history = pd.concat([oof,test],ignore_index=True)
    if not (final_history.cutoff + pd.to_timedelta(final_history.horizon*30,unit='m')).lt(query.timestamp.min()-pd.Timedelta(minutes=30)).all():
        raise ValueError('Исторические ответы пересекли конкурсный cutoff')
    final_factors = fit_route_adjustment(final_history)
    initial = query.y_pred
    changed = initial*query.route_id.map(final_factors).fillna(1.)
    old_total = initial.groupby(query.timestamp).sum()
    new_total = changed.groupby(query.timestamp).sum()
    if new_total.le(0).any() or not np.isfinite(new_total).all():
        raise ValueError('Некорректный объём после поправки')
    prediction = changed * query.timestamp.map(old_total/new_total)
    if not np.allclose(prediction.groupby(query.timestamp).sum(),old_total,rtol=1e-12,atol=1e-5):
        raise ValueError('Изменился объём горизонта')
    destination = OUT/'submission_route_bias_full_history.csv'
    query[['id']].assign(y_pred=prediction).to_csv(destination.with_suffix('.partial'),index=False)
    destination.with_suffix('.partial').replace(destination)
    print('Сабмит:', destination)
else:
    print('Для сборки CSV загрузите родительский сабмит:', PARENT)
print('Результаты:', OUT)
